# House target exceptions

Look up a `race_type` in `exceptions.csv` (written by `house_impute_targets_jungle_rcv.py`): its description, how many `targets.csv` rows it has and how many have a blank `adj_margin_2p_norm`, and the first `targets.csv` row it applies to. Set `CATEGORY` and run every cell.

In [1]:
import ast
from pathlib import Path

import pandas as pd

EXCEPTIONS_PATH = Path("exceptions.csv")
TARGETS_PATH = Path("../../data/processed/house/targets.csv")
KEY = ["year", "state", "district"]

exceptions = pd.read_csv(EXCEPTIONS_PATH).set_index("race_type")
# Applies to holds a list of (year, state, district); it is blank for dr, which covers nearly every row
exceptions["Applies to"] = exceptions["Applies to"].map(lambda s: ast.literal_eval(s) if pd.notna(s) else None)
targets = pd.read_csv(TARGETS_PATH, dtype_backend="numpy_nullable")  # vote counts stay integers despite blanks
exceptions[["Rows", "Blank"]]

,Rows,Blank
race_type,,
dr,3821,0
dem_uncontested,117,117
dem_vs_3p,112,112
gop_vs_3p,88,88
gop_uncontested,87,87
dem_vs_dem,44,44
dr_jungle,40,0
gop_vs_gop,14,14
dr_multi,12,0


Descriptions follow the `race_type` table in `README.md`.

In [2]:
RACE_TYPE_DESCRIPTIONS = {
    "dr": "Ordinary race: exactly one D and one R, possibly with third-party candidates. Target is the observed D vs R margin.",
    "dr_multi": "Ordinary race with more than one D or R, almost always a party-labeled write-in with a handful of votes "
                "next to the nominee. Each party's votes are summed; target is observed.",
    "dr_jungle": "All-candidate November round (LA 1998-2006 and 2012-2024, the TX specials of 1996 and 2006), often with "
                 "several candidates per party. Target is imputed from each party's combined first-round votes; the "
                 "runoff isn't used.",
    "dr_rcv": "Ranked-choice voting (ME from 2018, AK from 2022). Target is imputed from the first round, treated like a "
              "jungle round; for ME-2 2018, 2022 and 2024 the source rows aren't first-round counts.",
    "dr_writein": "One major party has only write-in candidates, so the race is effectively uncontested on the ballot and "
                  "its margin is near +/-100. Target is observed. Planned: treat as dem_uncontested / gop_uncontested, except 2006 TX-22 (a Republican write-in nominee in a real two-party race), which keeps its target.",
    "dem_vs_3p": "One Democrat and no Republican, against third-party or write-in candidates only. No target. Planned: imputed as an uncontested race (imputation.py).",
    "gop_vs_3p": "One Republican and no Democrat, against third-party or write-in candidates only. No target. Planned: imputed as an uncontested race (imputation.py).",
    "dem_uncontested": "A single named candidate, a Democrat; scattered write-in votes may exist. No target. Planned: imputed as an uncontested race (imputation.py).",
    "gop_uncontested": "A single named candidate, a Republican; scattered write-in votes may exist. No target. Planned: imputed as an uncontested race (imputation.py).",
    "dem_vs_dem": "Two or more Democrats and no Republican (CA/WA top-two, LA jungle). No target. Planned: imputed as an uncontested race (imputation.py).",
    "gop_vs_gop": "Two or more Republicans and no Democrat (CA/WA top-two, LA jungle). No target. Planned: imputed as an uncontested race (imputation.py).",
    "cross_endorsed": "The only major-party candidate was on both the D and R lines (NY 1976-1990); D/R votes are blank. "
                      "No target. Planned: imputed as an uncontested race (imputation.py).",
    "no_major_party": "No D or R candidate. No target. Planned: imputed as an uncontested race (imputation.py).",
}

In [3]:
def show(category: str) -> None:
    """Print category's description, counts, and the first targets.csv row it applies to."""
    if category not in exceptions.index:
        raise KeyError(f"{category!r} isn't in {EXCEPTIONS_PATH}; it has {list(exceptions.index)}")
    entry = exceptions.loc[category]
    print(f"{category}: {RACE_TYPE_DESCRIPTIONS[category]}")
    print(f"Rows {entry['Rows']}, blank adj_margin_2p_norm {entry['Blank']}\n")
    keys = entry["Applies to"]
    if keys is None:  # dr: take its first row in targets.csv
        row = targets[targets["race_type"] == category].iloc[0]
    else:
        year, state, district = keys[0]
        row = targets[(targets["year"] == year) & (targets["state"] == state)
                      & (targets["district"] == district)].iloc[0]
    print(f"First row, of {len(keys) if keys is not None else entry['Rows']}:")
    print(row.to_string())

In [ ]:
CATEGORY = "dr_writein"
show(CATEGORY)

dr_rcv: Ranked-choice voting (ME from 2018, AK from 2022). Target is imputed from the first round, treated like a jungle round; for ME-2 2018, 2022 and 2024 the source rows aren't first-round counts.
Rows 10, blank adj_margin_2p_norm 0

First row, of 10:
year                                                            2018
state                                                             ME
district                                                           1
dem_votes                                                     201195
gop_votes                                                     111188
3p_votes                                                       29670
total_votes                                                   342053
dem_pct                                                    58.819832
gop_pct                                                    32.506074
3p_pct                                                      8.674094
margin_votes                                           

## Every district of a category, with summary statistics

`category_table` returns each district of the category (every numeric `targets.csv` column that isn't blank throughout) followed by summary rows: `mean`, `median`, `std`, `min`, `q1`, `q3`, `max`, the outlier fences `lower_fence` / `upper_fence`, and `outliers`, the count of flagged rows. A value outside [Q1 − `OUTLIER_IQR`·IQR, Q3 + `OUTLIER_IQR`·IQR] of its column is an outlier; `outlier_columns` lists a district's, and they are highlighted below. A column whose values are all equal has IQR 0, so any other value in it is flagged.

In [11]:
OUTLIER_IQR = 1.5
STAT_ROWS = ["mean", "median", "std", "min", "q1", "q3", "max", "lower_fence", "upper_fence", "outliers"]


def category_rows(category: str) -> pd.DataFrame:
    """The targets.csv rows category applies to, in exceptions.csv's order (every row of its race_type for dr)."""
    if category not in exceptions.index:
        raise KeyError(f"{category!r} isn't in {EXCEPTIONS_PATH}; it has {list(exceptions.index)}")
    keys = exceptions.loc[category, "Applies to"]
    if keys is None:
        return targets[targets["race_type"] == category]
    return targets.set_index(KEY).loc[keys].reset_index()


def category_table(category: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    """(table, outlier): the category's districts, labeled 'year state-district', plus STAT_ROWS, and a mask of the
    district cells outside their column's fences."""
    rows = category_rows(category)
    numeric = rows.drop(columns=["year", "district"]).select_dtypes("number").astype("float64")
    numeric = numeric.dropna(axis=1, how="all")
    numeric.index = [f"{y} {s}-{d}" for y, s, d in rows[KEY].itertuples(index=False)]
    q1, q3 = numeric.quantile(0.25), numeric.quantile(0.75)
    lower, upper = q1 - OUTLIER_IQR * (q3 - q1), q3 + OUTLIER_IQR * (q3 - q1)
    outlier = numeric.lt(lower) | numeric.gt(upper)
    stats = pd.DataFrame({"mean": numeric.mean(), "median": numeric.median(), "std": numeric.std(),
                          "min": numeric.min(), "q1": q1, "q3": q3, "max": numeric.max(),
                          "lower_fence": lower, "upper_fence": upper, "outliers": outlier.sum()}).T
    table = pd.concat([numeric, stats])
    table["outlier_columns"] = outlier.apply(lambda r: ", ".join(r.index[r]), axis=1).reindex(table.index)
    return table, outlier.reindex(index=table.index, columns=table.columns, fill_value=False)


def show_table(category: str):
    """category_table as a styled table: outlier cells highlighted, summary rows shaded."""
    table, outlier = category_table(category)
    flagged = (table.loc[~table.index.isin(STAT_ROWS), "outlier_columns"] != "").sum()
    print(f"{category}: {len(table) - len(STAT_ROWS)} districts, {flagged} with at least one outlier")
    return (table.style
            .format(precision=2, thousands=",", na_rep="")
            .apply(lambda _: outlier.replace({True: "background-color: #f4c7a1", False: ""}), axis=None)
            .apply(lambda r: ["background-color: #e8e8e8" if r.name in STAT_ROWS else ""] * len(r), axis=1))

In [12]:
show_table(CATEGORY)

dr_rcv: 10 districts, 3 with at least one outlier


,dem_votes,gop_votes,3p_votes,total_votes,dem_pct,gop_pct,3p_pct,margin_votes,margin_raw,margin_2p_norm,dem_candidate_count,gop_candidate_count,3p_candidate_count,adj_margin_2p_norm,dem_votes_imputed,gop_votes_imputed,source_totalvotes,nonvote_votes,outlier_columns
2018 ME-1,"201,195.00","111,188.00","29,670.00","342,053.00",58.82,32.51,8.67,"90,007.00",26.31,28.81,1.00,1.00,1.00,28.81,"201,195.00","111,188.00","342,053.00",0.00,
2018 ME-2,"139,231.00","136,326.00",0.00,"275,557.00",50.53,49.47,0.00,"2,905.00",1.05,1.05,1.00,1.00,0.00,1.05,"139,231.00","136,326.00","281,371.00",0.00,
2020 ME-1,"271,004.00","165,008.00",15.00,"436,027.00",62.15,37.84,0.00,"105,996.00",24.31,24.31,1.00,1.00,0.00,24.31,"271,004.00","165,008.00","447,981.00","11,954.00",
2020 ME-2,"197,974.00","175,228.00",33.00,"373,235.00",53.04,46.95,0.01,"22,746.00",6.09,6.09,1.00,1.00,0.00,6.09,"197,974.00","175,228.00","380,324.00","7,089.00",
2022 AK-0,"128,553.00","129,379.00","5,678.00","263,610.00",48.77,49.08,2.15,-826.00,-0.31,-0.32,1.00,2.00,1.00,-0.32,"128,553.00","129,379.00","263,610.00",0.00,gop_candidate_count
2022 ME-1,"219,753.00","129,263.00",160.00,"349,176.00",62.93,37.02,0.05,"90,490.00",25.92,25.93,1.00,1.00,0.00,25.93,"219,753.00","129,263.00","358,138.00","8,962.00",
2022 ME-2,"165,136.00","146,142.00","22,048.00","333,326.00",49.54,43.84,6.61,"18,994.00",5.70,6.10,1.00,1.00,1.00,6.10,"165,136.00","146,142.00","656,104.00","322,778.00","source_totalvotes, nonvote_votes"
2024 AK-0,"156,245.00","159,550.00","13,760.00","329,555.00",47.41,48.41,4.18,"-3,305.00",-1.00,-1.05,2.00,1.00,1.00,-1.05,"156,245.00","159,550.00","329,555.00",0.00,dem_candidate_count
2024 ME-1,"249,798.00","154,849.00","20,883.00","425,530.00",58.70,36.39,4.91,"94,949.00",22.31,23.46,1.00,1.00,1.00,23.46,"249,798.00","154,849.00","425,530.00","14,037.00",
2024 ME-2,"197,151.00","194,445.00",0.00,"391,596.00",50.35,49.65,0.00,"2,706.00",0.69,0.69,1.00,1.00,0.00,0.69,"197,151.00","194,445.00","402,936.00","11,340.00",
